<a href="https://colab.research.google.com/github/jesildabraganca0511/NLP/blob/main/Word2Vec(from_scratch).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
def skipgram_pairs(docs,window=2):
  pairs=[]
  for doc in docs:
    for i,center in enumerate(doc):
      for j in range(max(0,i-window),min(len(doc),i+window+1)):
        if i==j:
          continue
        pairs.append((center,doc[j]))
  return pairs


In [5]:
skipgram_pairs([["the", "cat", "sat", "on", "mat"]], window=2)
[('the', 'cat'), ('the', 'sat'),
 ('cat', 'the'), ('cat', 'sat'), ('cat', 'on'),
 ('sat', 'the'), ('sat', 'cat'), ('sat', 'on'), ('sat', 'mat'),
 ...]

[('the', 'cat'),
 ('the', 'sat'),
 ('cat', 'the'),
 ('cat', 'sat'),
 ('cat', 'on'),
 ('sat', 'the'),
 ('sat', 'cat'),
 ('sat', 'on'),
 ('sat', 'mat'),
 Ellipsis]

In [6]:
import numpy as np

In [7]:
def init_embedding(vocab_size,dim,seed=42):
  rng=np.random.default_rng(seed)
  W = rng.normal(0, 0.1, size=(vocab_size, dim))
  W_prime = rng.normal(0, 0.1, size=(vocab_size, dim))
  return W, W_prime

# Negative Sampling

In [8]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20, 20)))

In [10]:
from typing import ValuesView
def train_pairs(W,W_prime,center_idx,context_idx,negative_indices,lr):
  v_c=W[center_idx]
  u_pos=W_prime[context_idx]
  u_neg=W_prime[negative_indices]

  # Forward
  pos_score=sigmoid(v_c @ u_pos)
  neg_score=sigmoid(v_c @ u_neg)

  # calculating error and gradient

  pos_error=pos_score-1
  gradient=pos_error*u_pos

  for i,value in enumerate(u_neg):
    gradient+=neg_score[i]*value




  # Adjusting matrix weights

  W[center_idx]-=lr*gradient
  W_prime[context_idx] -= lr * (pos_score - 1) * v_c
  for i, neg_idx in enumerate(negative_indices):
      W_prime[neg_idx] -= lr * neg_score[i] * v_c

